# Задача 07. Воронка конверсии по каналам привлечения

**Постановка задачи:** нужно сравнить каналы привлечения по пользовательской воронке:

`session → product_viewed → added_to_cart → checkout_started → purchase`

Для каждого канала посчитать количество сессий на каждом шаге и конверсии между шагами.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path('../data/marketplace.sqlite')
conn = sqlite3.connect(DB_PATH)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

print(f'База подключена: {DB_PATH.resolve()}')

In [ ]:
query = r"""
SELECT
    acquisition_channel,
    COUNT(session_id) AS sessions,
    SUM(product_viewed) AS product_views,
    SUM(added_to_cart) AS carts,
    SUM(checkout_started) AS checkouts,
    COUNT(order_id) AS purchases,
    ROUND(100.0 * SUM(product_viewed) / COUNT(session_id), 2) AS view_rate_pct,
    ROUND(100.0 * SUM(added_to_cart) / NULLIF(SUM(product_viewed), 0), 2) AS cart_to_view_pct,
    ROUND(100.0 * SUM(checkout_started) / NULLIF(SUM(added_to_cart), 0), 2) AS checkout_to_cart_pct,
    ROUND(100.0 * COUNT(order_id) / NULLIF(SUM(checkout_started), 0), 2) AS purchase_to_checkout_pct,
    ROUND(100.0 * COUNT(order_id) / COUNT(session_id), 2) AS session_to_purchase_pct
FROM sessions
GROUP BY acquisition_channel
ORDER BY session_to_purchase_pct DESC;
"""

df = pd.read_sql_query(query, conn)
df

**Комментарий стажёра:** общий `session_to_purchase_pct` показывает итоговую эффективность канала, а промежуточные конверсии помогают понять, на каком шаге пользователи чаще отваливаются.